# Ejemplo de uso del modulo con dataset Fousquare de Chile

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "Foursquare" 

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 100)
pd.set_option("display.max_colwidth", 120)

In [2]:
from __future__ import annotations

from typing import Iterable

import pandas as pd
import geopandas as gpd
import osmnx as ox


def read_foursquare_csv(path: str) -> pd.DataFrame:
    """
    Lee un archivo Foursquare separado por tab.
    """
    return pd.read_csv(path, sep="\t", dtype=str, low_memory=False)


def get_municipalities_gdf(municipalities: Iterable[str]) -> gpd.GeoDataFrame:
    """
    Obtiene el GeoDataFrame de comunas/municipios desde OSM.
    """
    amb = ox.geocoder.geocode_to_gdf(list(municipalities))
    return amb.to_crs("EPSG:4326")


def prepare_foursquare_triplevel_for_import(
    df_venues: pd.DataFrame,
    df_checkins: pd.DataFrame,
    municipalities: list[str],
) -> pd.DataFrame:
    """
    Preprocess manual nivel 1 para Foursquare.

    Qué hace:
    - filtra venues a las comunas indicadas,
    - cruza checkins con venues,
    - ordena por usuario y datetime,
    - construye viajes a partir de pares consecutivos de check-ins
      del mismo usuario,
    - devuelve una tabla con columnas:
        trip_id, user_id,
        origin_lat, origin_lon,
        destination_lat, destination_lon,
        origin_datetime, destination_datetime,
        origin_category, destination_category

    NO hace:
    - H3
    - correspondencias
    - limpieza de import
    - coerciones especiales de import
    """
    # 1) Comunas / municipios
    municipalities_gdf = get_municipalities_gdf(municipalities)

    # 2) Venues dentro del bbox
    venues = df_venues.copy()
    venues["lat"] = pd.to_numeric(venues["lat"], errors="coerce")
    venues["lon"] = pd.to_numeric(venues["lon"], errors="coerce")

    bbox = municipalities_gdf.total_bounds
    venues_bbox = venues[
        venues["lon"].between(bbox[0], bbox[2])
        & venues["lat"].between(bbox[1], bbox[3])
    ].copy()

    # 3) Filtrar venues que realmente caen dentro de las comunas
    venues_gdf = gpd.GeoDataFrame(
        venues_bbox[["venue_id", "lat", "lon", "category", "country"]].copy(),
        geometry=gpd.points_from_xy(venues_bbox["lon"], venues_bbox["lat"]),
        crs="EPSG:4326",
    )

    venues_in_area = gpd.sjoin(
        venues_gdf,
        municipalities_gdf[["geometry"]],
        predicate="within",
        how="inner",
    )

    venues_in_area = (
        venues_in_area
        .drop(columns=["index_right"], errors="ignore")
        .groupby("venue_id", as_index=False)
        .first()
    )

    # 4) Check-ins cuyo venue está dentro del área
    checkins = df_checkins.copy()
    checkins_in_area = checkins[checkins["venue_id"].isin(venues_in_area["venue_id"])].copy()

    # 5) Cruzar checkins con venues
    checkins_enriched = checkins_in_area.merge(
        venues_in_area[["venue_id", "lat", "lon", "category"]],
        on="venue_id",
        how="left",
    )

    # 6) Ordenar por usuario y datetime
    checkins_enriched = checkins_enriched.sort_values(
        by=["user_id", "datetime"],
        ascending=[True, True],
        kind="stable",
    ).reset_index(drop=True)

    # 7) Construir pares consecutivos por usuario
    origin = checkins_enriched.rename(
        columns={
            "lat": "origin_lat",
            "lon": "origin_lon",
            "datetime": "origin_datetime",
            "category": "origin_category",
            "venue_id": "origin_venue_id",
        }
    )[["user_id", "origin_venue_id", "origin_lat", "origin_lon", "origin_datetime", "origin_category"]]

    destination = checkins_enriched.rename(
        columns={
            "user_id": "user_id_dest",
            "lat": "destination_lat",
            "lon": "destination_lon",
            "datetime": "destination_datetime",
            "category": "destination_category",
            "venue_id": "destination_venue_id",
        }
    )[["user_id_dest", "destination_venue_id", "destination_lat", "destination_lon", "destination_datetime", "destination_category"]]

    trips = (
        origin.shift(1)
        .join(destination)
        .dropna(subset=["user_id", "user_id_dest"])
        .pipe(lambda x: x[x["user_id"] == x["user_id_dest"]])
        .drop(columns=["user_id_dest"])
        .reset_index(drop=True)
    )

    # 8) trip_id explícito
    trips.index.name = "trip_id"
    trips = trips.reset_index()

    # 9) Dejar exactamente las columnas pedidas
    trips = trips[
        [
            "trip_id",
            "user_id",
            "origin_lat",
            "origin_lon",
            "destination_lat",
            "destination_lon",
            "origin_datetime",
            "destination_datetime",
            "origin_category",
            "destination_category",
        ]
    ].copy()

    return trips

In [6]:
venues = read_foursquare_csv(DATA_PATH / "chile_POIs.csv")
checkins = read_foursquare_csv(DATA_PATH / "chile_Checkins.csv")

In [8]:
display(checkins)

,user_id,venue_id,datetime,utc_offset
0,44093,4dc0367c6a23e5a549e68d9b,Tue Apr 03 18:00:22 +0000 2012,-180
1,980044,4bb8ddd31261d13ab608e998,Tue Apr 03 18:00:27 +0000 2012,-180
2,640995,4f7200c9e4b0e30fde677e23,Tue Apr 03 18:00:28 +0000 2012,-180
3,468520,4b63374bf964a520396b2ae3,Tue Apr 03 18:00:38 +0000 2012,-180
4,402134,4daa468c43a1128196f512f3,Tue Apr 03 18:00:51 +0000 2012,-180
...,...,...,...,...
2209976,105555,4d32094f5017a0933059459b,Wed Jan 29 16:43:19 +0000 2014,-180
2209977,801725,4e8253e6f5b9398d5f9ba042,Wed Jan 29 16:43:27 +0000 2014,-180
2209978,1175638,4c2a17803492a59340e8b928,Wed Jan 29 16:43:45 +0000 2014,-180
2209979,2029127,4db1ec9a43a1bea42bd82943,Wed Jan 29 16:43:52 +0000 2014,-180


In [7]:
municipalities = '''Cerrillos
La Reina
Pudahuel
Cerro Navia
Las Condes
Quilicura
Conchalí
Lo Barnechea
Quinta Normal
El Bosque
Lo Espejo
Recoleta
Estación Central
Lo Prado
Renca
Huechuraba
Macul
San Miguel (Chile)
Independencia (Chile)
Maipú
San Joaquín (Chile)
La Cisterna
Ñuñoa
San Ramón (Chile)
La Florida
Pedro Aguirre Cerda
Santiago, Provincia de Santiago, Región Metropolitana de Santiago, Chile
La Pintana
Peñalolén
Vitacura
La Granja (Chile)
Providencia, Provincia de Santiago, Región Metropolitana de Santiago, Chile
Peñaflor (Chile)
San Bernardo (Chile)
Padre Hurtado
Puente Alto
'''.strip().split('\n')

In [9]:
df_foursquare_trips_for_import = prepare_foursquare_triplevel_for_import(
    df_venues=venues,
    df_checkins=checkins,
    municipalities=municipalities,
)

In [10]:
display(df_foursquare_trips_for_import)

,trip_id,user_id,origin_lat,origin_lon,destination_lat,destination_lon,origin_datetime,destination_datetime,origin_category,destination_category
0,0,1000072,-33.576827,-70.569537,-33.576824,-70.569595,Fri Apr 13 13:59:43 +0000 2012,Mon Apr 09 12:52:26 +0000 2012,Coworking Space,Parking
1,1,1000072,-33.576824,-70.569595,-33.518821,-70.600548,Mon Apr 09 12:52:26 +0000 2012,Mon Jun 04 23:36:21 +0000 2012,Parking,Subway
2,2,1000072,-33.518821,-70.600548,-33.579942,-70.580038,Mon Jun 04 23:36:21 +0000 2012,Sun Jun 10 05:52:50 +0000 2012,Subway,Home (private)
3,3,1000072,-33.579942,-70.580038,-33.574294,-70.555977,Sun Jun 10 05:52:50 +0000 2012,Thu Mar 28 14:43:07 +0000 2013,Home (private),Office
4,4,1000072,-33.574294,-70.555977,-33.518968,-70.599654,Thu Mar 28 14:43:07 +0000 2013,Tue Apr 03 22:35:21 +0000 2012,Office,Fried Chicken Joint
...,...,...,...,...,...,...,...,...,...,...
1265316,1265316,99998,-33.429728,-70.620044,-33.396967,-70.793753,Thu Jan 03 15:45:35 +0000 2013,Thu Mar 28 15:13:11 +0000 2013,Other Great Outdoors,Airport
1265317,1265317,99998,-33.396967,-70.793753,-33.404006,-70.609541,Thu Mar 28 15:13:11 +0000 2013,Tue Apr 30 22:16:15 +0000 2013,Airport,Scenic Lookout
1265318,1265318,99998,-33.404006,-70.609541,-33.444276,-70.614170,Tue Apr 30 22:16:15 +0000 2013,Tue Apr 30 23:39:09 +0000 2013,Scenic Lookout,South American Restaurant
1265319,1265319,99998,-33.444276,-70.614170,-33.396967,-70.793753,Tue Apr 30 23:39:09 +0000 2013,Tue Dec 11 19:56:37 +0000 2012,South American Restaurant,Airport


### Ahora probaremos la funcion de import trips

In [17]:
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError

#### Correccion minima necesaria

In [11]:
df_foursquare_trips_for_import = df_foursquare_trips_for_import.copy()
df_foursquare_trips_for_import["movement_id_src"] = df_foursquare_trips_for_import["trip_id"].astype(str)

#### Helpers para cargar e invertir el YAML de categorias

In [12]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import yaml

from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec


def load_yaml_file(path: str | Path) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def clean_domain_values(values: list[Any]) -> list[str]:
    out = []
    seen = set()

    for v in values:
        s = str(v).strip()
        if not s:
            continue
        if s not in seen:
            seen.add(s)
            out.append(s)

    return out


def clean_domain_dict(domains_raw: dict[str, list[Any]]) -> dict[str, list[str]]:
    return {
        k: clean_domain_values(v)
        for k, v in domains_raw.items()
    }


def invert_grouped_category_yaml(grouped_yaml: dict[str, list[str]]) -> dict[str, str]:
    """
    Convierte un YAML del tipo:
        Grupo -> [categorías finas...]
    a:
        categoría fina -> Grupo
    """
    inverted: dict[str, str] = {}

    for group_name, values in grouped_yaml.items():
        for raw_value in values:
            raw_value = str(raw_value).strip()
            if not raw_value:
                continue
            inverted[raw_value] = group_name

    return inverted

#### Schema base

In [13]:
BASE_GOLONDRINA_IMPORT_SCHEMA = TripSchema(
    version="1.1",
    fields={
        # required core
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
        "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
        "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),
        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        # base categóricos canónicos
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "walk", "bicycle", "scooter", "motorcycle", "car",
                    "taxi", "ride_hailing", "bus", "metro", "train", "other"
                ],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "home", "work", "education", "shopping",
                    "errand", "health", "leisure", "transfer", "other"
                ],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["night", "morning", "midday", "afternoon", "evening"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),

        # base no categóricos
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "mode_sequence": FieldSpec("mode_sequence", "string", required=False),
        "user_age_group": FieldSpec("user_age_group", "categorical", required=False),
        "income_quintile": FieldSpec("income_quintile", "categorical", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

#### Schema para Foursuare trips

In [14]:
def make_foursquare_trips_import_schema(foursquare_categories_yaml: str | Path) -> TripSchema:
    grouped = clean_domain_dict(load_yaml_file(foursquare_categories_yaml))
    reduced_groups = list(grouped.keys())

    extra_fields = {
        "origin_category": FieldSpec(
            "origin_category",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=reduced_groups,
                extendable=True,
            ),
        ),
        "destination_category": FieldSpec(
            "destination_category",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=reduced_groups,
                extendable=True,
            ),
        ),
    }

    return TripSchema(
        version=BASE_GOLONDRINA_IMPORT_SCHEMA.version,
        fields={**BASE_GOLONDRINA_IMPORT_SCHEMA.fields, **extra_fields},
        required=list(BASE_GOLONDRINA_IMPORT_SCHEMA.required),
    )

#### ImportOptions para Foursquare trips

In [15]:
FOURSQUARE_TRIPS_IMPORT_OPTIONS = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

#### Field Correspondence y Value correspondence para Foursquare trips

In [16]:
FOURSQUARE_TRIPS_FIELD_CORRESPONDENCE = {
    "user_id": "user_id",
    "movement_id": "movement_id_src",

    "origin_longitude": "origin_lon",
    "origin_latitude": "origin_lat",
    "destination_longitude": "destination_lon",
    "destination_latitude": "destination_lat",

    "origin_time_utc": "origin_datetime",
    "destination_time_utc": "destination_datetime",

    # extensiones fuente-específicas
    "origin_category": "origin_category",
    "destination_category": "destination_category",
}

def make_foursquare_trips_value_correspondence(
    foursquare_categories_yaml: str | Path,
) -> dict[str, dict[str, str]]:
    grouped = load_yaml_file(foursquare_categories_yaml)
    raw_to_group = invert_grouped_category_yaml(grouped)

    return {
        "origin_category": dict(raw_to_group),
        "destination_category": dict(raw_to_group),
    }

In [19]:
FOURSQUARE_TRIPS_SCHEMA = make_foursquare_trips_import_schema(
    DATA_PATH / "categories.yaml"
)

FOURSQUARE_TRIPS_OPTIONS = FOURSQUARE_TRIPS_IMPORT_OPTIONS

FOURSQUARE_TRIPS_FIELD_CORR = FOURSQUARE_TRIPS_FIELD_CORRESPONDENCE

FOURSQUARE_TRIPS_VALUE_CORR = make_foursquare_trips_value_correspondence(
    DATA_PATH / "categories.yaml"
)

In [21]:
trips_foursquare, report_foursquare = import_trips_from_dataframe(
    df_foursquare_trips_for_import.head(5000),
    schema=FOURSQUARE_TRIPS_SCHEMA,
    source_name="Foursquare trips level-1 manual preprocess",
    options=FOURSQUARE_TRIPS_OPTIONS,
    field_correspondence=FOURSQUARE_TRIPS_FIELD_CORR,
    value_correspondence=FOURSQUARE_TRIPS_VALUE_CORR,
    provenance={
        "source": {
            "name": "Foursquare",
            "entity": "trips",
            "version": "checkins-pois-level1",
        },
        "notes": ["preprocess manual nivel 1: pares consecutivos de check-ins"],
    },
    h3_resolution=8,
)

In [22]:
display(trips_foursquare.data)

,trip_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_time_utc,destination_time_utc,origin_category,destination_category,movement_id,origin_h3_index,destination_h3_index,movement_seq
0,0,1000072,-33.576827,-70.569537,-33.576824,-70.569595,2012-04-13 13:59:43+00:00,2012-04-09 12:52:26+00:00,Business and Professional Services,Travel and Transportation,0,88b2c5734bfffff,88b2c5734bfffff,0
1,1,1000072,-33.576824,-70.569595,-33.518821,-70.600548,2012-04-09 12:52:26+00:00,2012-06-04 23:36:21+00:00,Travel and Transportation,Subway,1,88b2c5734bfffff,88b2c5093dfffff,0
2,2,1000072,-33.518821,-70.600548,-33.579942,-70.580038,2012-06-04 23:36:21+00:00,2012-06-10 05:52:50+00:00,Subway,Community and Government,2,88b2c5093dfffff,88b2c57341fffff,0
3,3,1000072,-33.579942,-70.580038,-33.574294,-70.555977,2012-06-10 05:52:50+00:00,2013-03-28 14:43:07+00:00,Community and Government,Business and Professional Services,3,88b2c57341fffff,88b2c509b3fffff,0
4,4,1000072,-33.574294,-70.555977,-33.518968,-70.599654,2013-03-28 14:43:07+00:00,2012-04-03 22:35:21+00:00,Business and Professional Services,Dining and Drinking,4,88b2c509b3fffff,88b2c50939fffff,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,1004649,-33.451316,-70.649214,-33.451185,-70.648549,2012-12-30 11:24:16+00:00,2012-12-30 11:24:39+00:00,Community and Government,Landmarks and Outdoors,4995,88b2c554e9fffff,88b2c554c5fffff,0
4996,4996,1004649,-33.451185,-70.648549,-33.451403,-70.651421,2012-12-30 11:24:39+00:00,2012-12-30 11:29:48+00:00,Landmarks and Outdoors,Community and Government,4996,88b2c554c5fffff,88b2c554e9fffff,0
4997,4997,1004649,-33.451403,-70.651421,-33.428174,-70.651912,2012-12-30 11:29:48+00:00,2012-12-30 15:38:33+00:00,Community and Government,Travel and Transportation,4997,88b2c554e9fffff,88b2c55419fffff,0
4998,4998,1004649,-33.428174,-70.651912,-33.453753,-70.652301,2012-12-30 15:38:33+00:00,2012-12-30 20:13:27+00:00,Travel and Transportation,Drugstore / Pharmacy,4998,88b2c55419fffff,88b2c554e9fffff,0


# Nivel 3

## Para datos de Foursquare - Trips / viajes

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [4]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_foursquare.make_foursquare_trips_profile import (
    read_foursquare_csv,
    make_foursquare_trips_profile,
)
from scripts.source_profiles.factories_foursquare.trips_defaults import (
    FOURSQUARE_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

venues = read_foursquare_csv(DATA_PATH / "chile_POIs.csv")
checkins = read_foursquare_csv(DATA_PATH / "chile_Checkins.csv")

municipalities = [
    "Cerrillos, Chile",
    "La Reina, Chile",
    "Pudahuel, Chile",
    "Providencia, Chile",
    "Santiago, Chile",
]

profile = make_foursquare_trips_profile(
    df_venues=venues,
    municipalities=municipalities,
    categories_yaml=DATA_PATH / "categories.yaml",
)

df_test = profile.preprocess(checkins.head(1000))
display(df_test.head())

# inspección clara
profile.schema_override
profile.default_options
profile.default_field_correspondence
profile.default_value_correspondence

,trip_id,user_id,origin_lat,origin_lon,destination_lat,destination_lon,origin_datetime,destination_datetime,origin_category,destination_category,movement_id_src
0,0,1000325,-33.436643,-70.608324,-33.436819,-70.608780,Tue Apr 03 18:05:58 +0000 2012,Tue Apr 03 18:42:15 +0000 2012,College Bookstore,College Quad,0
1,1,1124504,-33.443492,-70.649943,-33.440616,-70.651421,Tue Apr 03 18:40:57 +0000 2012,Tue Apr 03 19:37:22 +0000 2012,Subway,Home (private),1
2,2,1247956,-33.449428,-70.657936,-33.440616,-70.651421,Tue Apr 03 18:20:32 +0000 2012,Tue Apr 03 19:39:39 +0000 2012,University,Home (private),2
3,3,1319570,-33.436599,-70.635073,-33.460207,-70.752193,Tue Apr 03 18:25:01 +0000 2012,Tue Apr 03 19:40:59 +0000 2012,Plaza,Grocery Store,3
4,4,1319570,-33.460207,-70.752193,-33.466732,-70.746583,Tue Apr 03 19:40:59 +0000 2012,Tue Apr 03 19:41:57 +0000 2012,Grocery Store,Home (private),4


{'origin_category': {'Arts and Entertainment': 'Arts and Entertainment',
  'Amusement Park': 'Arts and Entertainment',
  'Attraction': 'Arts and Entertainment',
  'Aquarium': 'Arts and Entertainment',
  'Arcade': 'Arts and Entertainment',
  'Art Gallery': 'Arts and Entertainment',
  'Bingo Center': 'Arts and Entertainment',
  'Bowling Alley': 'Arts and Entertainment',
  'Carnival': 'Arts and Entertainment',
  'Casino': 'Arts and Entertainment',
  'Circus': 'Arts and Entertainment',
  'Comedy Club': 'Arts and Entertainment',
  'Country Club': 'Arts and Entertainment',
  'Country Dance Club': 'Arts and Entertainment',
  'Dance Hall': 'Arts and Entertainment',
  'Disc Golf': 'Arts and Entertainment',
  'Disc Golf Course': 'Arts and Entertainment',
  'Escape Room': 'Arts and Entertainment',
  'Exhibit': 'Arts and Entertainment',
  'Fair': 'Arts and Entertainment',
  'Gaming Cafe': 'Arts and Entertainment',
  'General Entertainment': 'Arts and Entertainment',
  'Go Kart Track': 'Arts and En

In [6]:
trips_foursquare, report_foursquare = import_trips_from_profile(
    profile=profile,
    df=checkins.head(50000),
    source_name="Foursquare trips level-3 factory",
    provenance=FOURSQUARE_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_foursquare.data)
display(report_foursquare.issues)

,trip_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_time_utc,destination_time_utc,origin_category,destination_category,movement_id,origin_h3_index,destination_h3_index,movement_seq
0,0,1000231,-33.443361,-70.648043,-33.445962,-70.656753,2012-04-08 04:29:02+00:00,2012-04-08 04:32:45+00:00,Community and Government,Business and Professional Services,0,88b2c55413fffff,88b2c55417fffff,0
1,1,1000325,-33.439945,-70.639744,-33.425460,-70.613723,2012-04-09 12:31:44+00:00,2012-04-09 12:42:43+00:00,Subway,Subway,1,88b2c554cdfffff,88b2c55687fffff,0
2,2,1000325,-33.425460,-70.613723,-33.436724,-70.608058,2012-04-09 12:42:43+00:00,2012-04-09 13:12:50+00:00,Subway,Community and Government,2,88b2c55687fffff,88b2c556b1fffff,0
3,3,1000325,-33.436724,-70.608058,-33.436554,-70.608317,2012-04-09 13:12:50+00:00,2012-04-09 15:18:34+00:00,Community and Government,Community and Government,3,88b2c556b1fffff,88b2c556b1fffff,0
4,4,1000325,-33.436554,-70.608317,-33.436819,-70.608780,2012-04-09 15:18:34+00:00,2012-04-09 15:23:26+00:00,Community and Government,Community and Government,4,88b2c556b1fffff,88b2c556b1fffff,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7398,7398,999095,-33.439964,-70.551578,-33.440054,-70.551474,2012-04-03 18:14:48+00:00,2012-04-03 18:15:05+00:00,Community and Government,School,7398,88b2c50b5dfffff,88b2c50b5dfffff,0
7399,7399,999127,-33.441541,-70.649228,-33.442134,-70.665452,2012-04-09 15:33:33+00:00,2012-04-04 15:39:09+00:00,Dining and Drinking,Community and Government,7399,88b2c55413fffff,88b2c55415fffff,0
7400,7400,999127,-33.442134,-70.665452,-33.439886,-70.650925,2012-04-04 15:39:09+00:00,2012-04-04 18:11:08+00:00,Community and Government,Record Shop,7400,88b2c55415fffff,88b2c55411fffff,0
7401,7401,999127,-33.439886,-70.650925,-33.442400,-70.648999,2012-04-04 18:11:08+00:00,2012-04-04 18:23:59+00:00,Record Shop,Retail,7401,88b2c55411fffff,88b2c55413fffff,0


[Issue(level='warning', code='SCH.DOMAIN.MISSING_FOR_CATEGORICAL', message="El campo categórico 'user_age_group' no tiene DomainSpec asociado; se degradará a texto para permitir el import.", field='user_age_group', source_field=None, row_count=None, details={'field': 'user_age_group', 'dtype': 'categorical', 'fallback_dtype': 'string', 'action': 'fallback_dtype'}),
 Issue(level='warning', code='SCH.DOMAIN.MISSING_FOR_CATEGORICAL', message="El campo categórico 'income_quintile' no tiene DomainSpec asociado; se degradará a texto para permitir el import.", field='income_quintile', source_field=None, row_count=None, details={'field': 'income_quintile', 'dtype': 'categorical', 'fallback_dtype': 'string', 'action': 'fallback_dtype'}),
 Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='mode', source_field=None, row_count=None, details={'field': 'mode', 'source_co

### 2) Uso más personalizado, con decisiones específicas

In [7]:
from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.types import FieldCorrespondence, ValueCorrespondence
from scripts.source_profiles.factories_foursquare.trips_defaults import (
    load_yaml_file,
    clean_domain_values,
    clean_domain_dict,
    invert_grouped_category_yaml,
)

# -------------------------------------------------------------------------
# Objetos custom independientes
# -------------------------------------------------------------------------

def make_foursquare_trips_custom_schema(foursquare_categories_yaml: str | Path) -> TripSchema:
    grouped = clean_domain_dict(load_yaml_file(foursquare_categories_yaml))
    reduced_groups = list(grouped.keys())

    return TripSchema(
        version="1.1-foursquare-custom",
        fields={
            "movement_id": FieldSpec("movement_id", "string", required=True),
            "user_id": FieldSpec("user_id", "string", required=True),
            "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
            "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
            "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
            "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
            "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
            "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),
            "trip_id": FieldSpec("trip_id", "string", required=True),
            "movement_seq": FieldSpec("movement_seq", "int", required=True),
            "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
            "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
            "origin_category": FieldSpec(
                "origin_category",
                "categorical",
                required=False,
                domain=DomainSpec(values=reduced_groups, extendable=False),
            ),
            "destination_category": FieldSpec(
                "destination_category",
                "categorical",
                required=False,
                domain=DomainSpec(values=reduced_groups, extendable=False),
            ),
        },
        required=[
            "movement_id",
            "user_id",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "trip_id",
            "movement_seq",
        ],
    )


FOURSQUARE_TRIPS_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
        "origin_time_utc",
        "destination_time_utc",
        "origin_category",
        "destination_category",
    ],
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

FOURSQUARE_TRIPS_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "user_id": "user_id",
    "movement_id": "movement_id_src",
    "origin_longitude": "origin_lon",
    "origin_latitude": "origin_lat",
    "destination_longitude": "destination_lon",
    "destination_latitude": "destination_lat",
    "origin_time_utc": "origin_datetime",
    "destination_time_utc": "destination_datetime",
    "origin_category": "origin_category",
    "destination_category": "destination_category",
}

def make_foursquare_trips_custom_value_correspondence(
    foursquare_categories_yaml: str | Path,
) -> ValueCorrespondence:
    grouped = load_yaml_file(foursquare_categories_yaml)
    raw_to_group = invert_grouped_category_yaml(grouped)

    return {
        "origin_category": dict(raw_to_group),
        "destination_category": dict(raw_to_group),
    }

FOURSQUARE_TRIPS_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "Foursquare",
        "profile": "FOURSQUARE_TRIPS_CUSTOM",
        "entity": "trips",
        "version": "checkins-pois-level3",
    },
    "notes": [
        "factory nivel 3 Foursquare custom",
        "schema y mappings definidos explícitamente",
    ],
}

In [10]:
custom_schema = make_foursquare_trips_custom_schema(DATA_PATH / "categories.yaml")
custom_value_corr = make_foursquare_trips_custom_value_correspondence(DATA_PATH / "categories.yaml")

profile_custom = make_foursquare_trips_profile(
    df_venues=venues,
    municipalities=municipalities,
    categories_yaml=DATA_PATH / "categories.yaml",
    schema_override=custom_schema,
    field_correspondence_override=FOURSQUARE_TRIPS_CUSTOM_FIELD_CORRESPONDENCE,
    value_correspondence_override=custom_value_corr,
    options_override={
        "keep_extra_fields": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.selected_fields,
        "strict": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.strict,
        "strict_domains": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.strict_domains,
        "single_stage": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.single_stage,
        "source_timezone": FOURSQUARE_TRIPS_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="FOURSQUARE_TRIPS_CUSTOM",
)

df_test = profile_custom.preprocess(checkins.head(1000))
display(df_test.head())

# inspección clara
profile_custom.schema_override
profile_custom.default_options
profile_custom.default_field_correspondence
profile_custom.default_value_correspondence


,trip_id,user_id,origin_lat,origin_lon,destination_lat,destination_lon,origin_datetime,destination_datetime,origin_category,destination_category,movement_id_src
0,0,1000325,-33.436643,-70.608324,-33.436819,-70.608780,Tue Apr 03 18:05:58 +0000 2012,Tue Apr 03 18:42:15 +0000 2012,College Bookstore,College Quad,0
1,1,1124504,-33.443492,-70.649943,-33.440616,-70.651421,Tue Apr 03 18:40:57 +0000 2012,Tue Apr 03 19:37:22 +0000 2012,Subway,Home (private),1
2,2,1247956,-33.449428,-70.657936,-33.440616,-70.651421,Tue Apr 03 18:20:32 +0000 2012,Tue Apr 03 19:39:39 +0000 2012,University,Home (private),2
3,3,1319570,-33.436599,-70.635073,-33.460207,-70.752193,Tue Apr 03 18:25:01 +0000 2012,Tue Apr 03 19:40:59 +0000 2012,Plaza,Grocery Store,3
4,4,1319570,-33.460207,-70.752193,-33.466732,-70.746583,Tue Apr 03 19:40:59 +0000 2012,Tue Apr 03 19:41:57 +0000 2012,Grocery Store,Home (private),4


{'origin_category': {'Arts and Entertainment': 'Arts and Entertainment',
  'Amusement Park': 'Arts and Entertainment',
  'Attraction': 'Arts and Entertainment',
  'Aquarium': 'Arts and Entertainment',
  'Arcade': 'Arts and Entertainment',
  'Art Gallery': 'Arts and Entertainment',
  'Bingo Center': 'Arts and Entertainment',
  'Bowling Alley': 'Arts and Entertainment',
  'Carnival': 'Arts and Entertainment',
  'Casino': 'Arts and Entertainment',
  'Circus': 'Arts and Entertainment',
  'Comedy Club': 'Arts and Entertainment',
  'Country Club': 'Arts and Entertainment',
  'Country Dance Club': 'Arts and Entertainment',
  'Dance Hall': 'Arts and Entertainment',
  'Disc Golf': 'Arts and Entertainment',
  'Disc Golf Course': 'Arts and Entertainment',
  'Escape Room': 'Arts and Entertainment',
  'Exhibit': 'Arts and Entertainment',
  'Fair': 'Arts and Entertainment',
  'Gaming Cafe': 'Arts and Entertainment',
  'General Entertainment': 'Arts and Entertainment',
  'Go Kart Track': 'Arts and En

In [12]:
trips_foursquare_custom, report_foursquare_custom = import_trips_from_profile(
    profile=profile_custom,
    df=checkins.head(50000),
    source_name="Foursquare trips level-3 custom factory",
    provenance=FOURSQUARE_TRIPS_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(trips_foursquare_custom.data)
display(report_foursquare_custom.issues)

,trip_id,user_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_time_utc,destination_time_utc,origin_category,destination_category,movement_id,origin_h3_index,destination_h3_index,movement_seq
0,0,1000231,-33.443361,-70.648043,-33.445962,-70.656753,2012-04-08 04:29:02+00:00,2012-04-08 04:32:45+00:00,Community and Government,Business and Professional Services,0,88b2c55413fffff,88b2c55417fffff,0
1,1,1000325,-33.439945,-70.639744,-33.425460,-70.613723,2012-04-09 12:31:44+00:00,2012-04-09 12:42:43+00:00,unknown,unknown,1,88b2c554cdfffff,88b2c55687fffff,0
2,2,1000325,-33.425460,-70.613723,-33.436724,-70.608058,2012-04-09 12:42:43+00:00,2012-04-09 13:12:50+00:00,unknown,Community and Government,2,88b2c55687fffff,88b2c556b1fffff,0
3,3,1000325,-33.436724,-70.608058,-33.436554,-70.608317,2012-04-09 13:12:50+00:00,2012-04-09 15:18:34+00:00,Community and Government,Community and Government,3,88b2c556b1fffff,88b2c556b1fffff,0
4,4,1000325,-33.436554,-70.608317,-33.436819,-70.608780,2012-04-09 15:18:34+00:00,2012-04-09 15:23:26+00:00,Community and Government,Community and Government,4,88b2c556b1fffff,88b2c556b1fffff,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7398,7398,999095,-33.439964,-70.551578,-33.440054,-70.551474,2012-04-03 18:14:48+00:00,2012-04-03 18:15:05+00:00,Community and Government,unknown,7398,88b2c50b5dfffff,88b2c50b5dfffff,0
7399,7399,999127,-33.441541,-70.649228,-33.442134,-70.665452,2012-04-09 15:33:33+00:00,2012-04-04 15:39:09+00:00,Dining and Drinking,Community and Government,7399,88b2c55413fffff,88b2c55415fffff,0
7400,7400,999127,-33.442134,-70.665452,-33.439886,-70.650925,2012-04-04 15:39:09+00:00,2012-04-04 18:11:08+00:00,Community and Government,unknown,7400,88b2c55415fffff,88b2c55411fffff,0
7401,7401,999127,-33.439886,-70.650925,-33.442400,-70.648999,2012-04-04 18:11:08+00:00,2012-04-04 18:23:59+00:00,unknown,Retail,7401,88b2c55411fffff,88b2c55413fffff,0


[Issue(level='warning', code='DOM.POLICY.FIELD_NOT_EXTENDABLE', message="El campo 'origin_category' no admite extensión de dominio (extendable=False); los valores fuera de dominio se mapearán a unknown.", field='origin_category', source_field=None, row_count=None, details={'field': 'origin_category', 'strict_domains': False, 'domain_extendable': False, 'action': 'map_to_unknown'}),
 Issue(level='warning', code='DOM.POLICY.FIELD_NOT_EXTENDABLE', message="El campo 'destination_category' no admite extensión de dominio (extendable=False); los valores fuera de dominio se mapearán a unknown.", field='destination_category', source_field=None, row_count=None, details={'field': 'destination_category', 'strict_domains': False, 'domain_extendable': False, 'action': 'map_to_unknown'}),
 Issue(level='warning', code='DOM.POLICY.FIELD_NOT_EXTENDABLE', message="El campo 'origin_category' no admite extensión de dominio (extendable=False); los valores fuera de dominio se mapearán a unknown.", field='ori